# 07 - Max Pain

The strike where option buyers would lose the most (i.e. the price
where the most options expire worthless).

Price tends to gravitate toward max pain near expiry.


## Setup

In [ ]:
import sys, sqlite3
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / "database").exists():
    _p = _p.parent
sys.path.insert(0, str(_p))

OPT_DB = _p / "database" / "options.db"
MKT_DB = _p / "database" / "market.db"

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

def opt(q, **p):
    with sqlite3.connect(OPT_DB) as c:
        return pd.read_sql_query(q, c, params=p)

def mkt(q, **p):
    with sqlite3.connect(MKT_DB) as c:
        return pd.read_sql_query(q, c, params=p)

print("options.db:", OPT_DB)
print("market.db :", MKT_DB)

## 1. Load chain

In [ ]:
oc = opt("""
    SELECT * FROM option_chain
    WHERE underlying='ETH'
      AND ts = (SELECT MAX(ts) FROM option_chain WHERE underlying='ETH')
""")
print("rows:", len(oc))

## 2. OI by strike

In [ ]:
if not oc.empty:
    by_strike = oc.groupby("strike")["oi"].sum().sort_index()

    fig, ax = plt.subplots()
    ax.bar(by_strike.index, by_strike.values, width=20, color="steelblue")
    ax.set_title("total OI by strike")
    ax.set_xlabel("strike")
    ax.set_ylabel("OI")
    plt.tight_layout()
    plt.show()

    mp = int(by_strike.idxmax())
    print(f"max pain (highest OI strike) = {mp}")

## 3. Put vs call OI by strike

In [ ]:
if not oc.empty:
    pivot = oc.pivot_table(index="strike", columns="kind",
                           values="oi", aggfunc="sum").fillna(0)
    pivot.plot.bar(stacked=True, figsize=(14, 4), color={"C": "seagreen", "P": "crimson"})
    plt.title("OI by strike — calls vs puts")
    plt.xlabel("strike")
    plt.tight_layout()
    plt.show()

## 4. Max pain over time

In [ ]:
s = opt("SELECT * FROM option_summary WHERE underlying='ETH' ORDER BY ts")
if not s.empty:
    s["date_dt"] = pd.to_datetime(s.ts, unit="ms")
    fig, ax = plt.subplots()
    ax.plot(s.date_dt, s.max_pain, "o-", color="purple", label="max pain")
    ax.set_title("max pain over time")
    ax.set_ylabel("strike")
    ax.legend()
    plt.tight_layout()
    plt.show()